# 11 — Retrieval-Augmented Multilingual Crop-Disease Advisory

This notebook creates a retrieval-grounded advisory layer for the Main19 crop-disease classifier.

## Purpose

The system combines:

1. A model prediction and calibrated confidence.
2. A curated crop–disease knowledge base.
3. Retrieval of the most relevant advisory entry.
4. A safety-aware response template in English, Hindi, and Telugu.

## Safety principles

- The image-model result is preliminary, not a confirmed diagnosis.
- PlantDoc evaluation showed substantial external domain shift.
- Low-confidence predictions should trigger abstention and expert-review guidance.
- No pesticide brand names, dosages, or location-specific chemical advice are provided.
- Management guidance prioritizes monitoring, sanitation, cultural practices,
  and consultation with local agricultural extension services.

In [ ]:
from pathlib import Path
import json
import re
import textwrap
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

WORKING_ROOT = Path("/kaggle/working")

RESULTS_DIR = (
    WORKING_ROOT
    / "results"
    / "notebook_11_rag_multilingual_advisory"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# This threshold is a policy threshold for advisory behavior.
# It is NOT assumed to solve PlantDoc domain shift.
HIGH_CONFIDENCE_THRESHOLD = 0.90
MEDIUM_CONFIDENCE_THRESHOLD = 0.70

SUPPORTED_LANGUAGES = {
    "en": "English",
    "hi": "Hindi",
    "te": "Telugu",
}

print("Results directory:", RESULTS_DIR)
print("Supported languages:", SUPPORTED_LANGUAGES)

In [ ]:
MAIN19_LABELS = [
    "bell_pepper__bacterial_spot",
    "bell_pepper__healthy",

    "corn__cercospora_leaf_spot_gray_leaf_spot",
    "corn__common_rust",
    "corn__healthy",
    "corn__northern_leaf_blight",

    "potato__early_blight",
    "potato__healthy",
    "potato__late_blight",

    "tomato__bacterial_spot",
    "tomato__early_blight",
    "tomato__healthy",
    "tomato__late_blight",
    "tomato__leaf_mold",
    "tomato__septoria_leaf_spot",
    "tomato__spider_mites_two_spotted_spider_mite",
    "tomato__target_spot",
    "tomato__tomato_mosaic_virus",
    "tomato__tomato_yellow_leaf_curl_virus",
]

assert len(MAIN19_LABELS) == 19

def split_main19_label(label):
    crop, disease = label.split("__", maxsplit=1)
    return crop, disease

main19_labels_df = pd.DataFrame(
    [
        {
            "label": label,
            "crop": split_main19_label(label)[0],
            "disease": split_main19_label(label)[1],
        }
        for label in MAIN19_LABELS
    ]
)

display(main19_labels_df)

In [ ]:
# This is a concise, project-specific, retrieval knowledge base.
# Guidance is educational and does not include pesticide product names,
# doses, or location-specific chemical recommendations.

knowledge_base = [
    {
        "label": "bell_pepper__bacterial_spot",
        "crop_en": "Bell pepper",
        "condition_en": "Bacterial spot",
        "symptoms_en": (
            "Small water-soaked or dark spots may develop on leaves. "
            "Spots can enlarge, become angular, and may lead to yellowing "
            "or leaf drop under favourable wet conditions."
        ),
        "monitor_en": (
            "Check new leaves and nearby plants for spreading spots. "
            "Record whether symptoms increase after rain, overhead irrigation, "
            "or handling plants while foliage is wet."
        ),
        "immediate_en": (
            "Avoid working in the crop when foliage is wet. Remove severely "
            "affected plant material only when practical, and avoid moving it "
            "between fields. Improve airflow and reduce prolonged leaf wetness."
        ),
        "prevention_en": (
            "Use clean planting material, sanitize tools, rotate away from "
            "susceptible hosts where feasible, and avoid overhead irrigation "
            "when alternatives are available."
        ),
        "escalate_en": (
            "Seek local extension or plant-health advice if symptoms spread "
            "rapidly, occur in seedlings, or affect a large proportion of plants."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "bell_pepper__healthy",
        "crop_en": "Bell pepper",
        "condition_en": "No obvious disease pattern detected",
        "symptoms_en": (
            "The classifier prediction is consistent with a healthy bell pepper leaf, "
            "but a single image cannot rule out early disease, nutritional stress, "
            "or hidden symptoms."
        ),
        "monitor_en": (
            "Inspect both upper and lower leaf surfaces and observe several plants "
            "across the field or garden."
        ),
        "immediate_en": (
            "Continue regular monitoring and maintain good field hygiene."
        ),
        "prevention_en": (
            "Use clean tools, avoid unnecessary handling of wet plants, and maintain "
            "appropriate irrigation, spacing, nutrition, and weed management."
        ),
        "escalate_en": (
            "Request expert support if new spots, yellowing, wilting, or rapid "
            "decline appear."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "corn__cercospora_leaf_spot_gray_leaf_spot",
        "crop_en": "Corn",
        "condition_en": "Gray leaf spot",
        "symptoms_en": (
            "Lesions are often rectangular or elongated and may appear gray to tan. "
            "They can expand along leaf veins and reduce green leaf area."
        ),
        "monitor_en": (
            "Inspect lower and middle leaves first, then assess whether lesions are "
            "spreading upward. Note recent humidity, rainfall, residue, and crop history."
        ),
        "immediate_en": (
            "Mark affected areas and monitor disease progression. Avoid moving "
            "potentially contaminated crop residue or tools between fields."
        ),
        "prevention_en": (
            "Use locally suitable resistant hybrids where available, rotate crops, "
            "and manage infected residue according to local agronomic guidance."
        ),
        "escalate_en": (
            "Consult local extension personnel if lesions are widespread before key "
            "growth stages or if disease pressure is increasing quickly."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "corn__common_rust",
        "crop_en": "Corn",
        "condition_en": "Common rust",
        "symptoms_en": (
            "Small reddish-brown or orange pustules may occur on either leaf surface. "
            "Severe infections can cause extensive leaf yellowing or drying."
        ),
        "monitor_en": (
            "Check multiple plants and both leaf surfaces. Track whether pustules are "
            "increasing and whether symptoms are concentrated in humid areas."
        ),
        "immediate_en": (
            "Monitor spread and maintain field records. Avoid unnecessary movement "
            "through wet foliage."
        ),
        "prevention_en": (
            "Use resistant hybrids when locally available and maintain balanced "
            "crop nutrition and good field management."
        ),
        "escalate_en": (
            "Ask a local crop adviser for confirmation when rust is widespread or "
            "appears early in a susceptible crop."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "corn__healthy",
        "crop_en": "Corn",
        "condition_en": "No obvious disease pattern detected",
        "symptoms_en": (
            "The prediction is consistent with a healthy corn leaf, but image-based "
            "screening cannot exclude early infection or non-disease stress."
        ),
        "monitor_en": (
            "Continue routine scouting across several locations in the field."
        ),
        "immediate_en": (
            "Maintain normal crop monitoring and record any emerging symptoms."
        ),
        "prevention_en": (
            "Use clean seed, suitable hybrids, crop rotation, field sanitation, "
            "and locally recommended agronomic practices."
        ),
        "escalate_en": (
            "Seek advice if symptoms appear suddenly, spread quickly, or affect multiple plants."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "corn__northern_leaf_blight",
        "crop_en": "Corn",
        "condition_en": "Northern leaf blight",
        "symptoms_en": (
            "Long, cigar-shaped gray-green to tan lesions may appear on leaves. "
            "Lesions can enlarge and merge under favourable conditions."
        ),
        "monitor_en": (
            "Inspect lower leaves and assess whether long lesions are increasing "
            "toward upper canopy leaves."
        ),
        "immediate_en": (
            "Monitor disease progression and avoid spreading residue or wet foliage "
            "between fields."
        ),
        "prevention_en": (
            "Use resistant hybrids where appropriate, rotate crops, and manage crop "
            "residue according to locally suitable practices."
        ),
        "escalate_en": (
            "Contact a local adviser if symptoms occur early, spread rapidly, or "
            "threaten a large area of the crop."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "potato__early_blight",
        "crop_en": "Potato",
        "condition_en": "Early blight",
        "symptoms_en": (
            "Older leaves may develop brown spots with concentric ring patterns, "
            "often surrounded by yellowing."
        ),
        "monitor_en": (
            "Check older leaves first and estimate how many plants show expanding spots."
        ),
        "immediate_en": (
            "Remove heavily affected leaves only if practical and hygienic. Reduce "
            "prolonged leaf wetness and avoid unnecessary handling of wet plants."
        ),
        "prevention_en": (
            "Use clean seed tubers, rotate crops, manage volunteer plants and debris, "
            "and maintain balanced crop nutrition."
        ),
        "escalate_en": (
            "Seek local advice if symptoms spread quickly, move to younger leaves, "
            "or threaten a substantial portion of the canopy."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "potato__healthy",
        "crop_en": "Potato",
        "condition_en": "No obvious disease pattern detected",
        "symptoms_en": (
            "The prediction is consistent with a healthy potato leaf, but continued "
            "field scouting is still necessary."
        ),
        "monitor_en": (
            "Inspect lower and upper canopy leaves on multiple plants."
        ),
        "immediate_en": (
            "Continue routine scouting and keep irrigation and field hygiene practices consistent."
        ),
        "prevention_en": (
            "Use healthy seed tubers, crop rotation, sanitation, and locally recommended "
            "soil, nutrition, and irrigation management."
        ),
        "escalate_en": (
            "Consult an adviser if dark spots, blight-like lesions, wilting, or rapid "
            "foliage decline develops."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "potato__late_blight",
        "crop_en": "Potato",
        "condition_en": "Late blight",
        "symptoms_en": (
            "Leaves may show irregular water-soaked, dark green to brown lesions. "
            "Under humid conditions, pale growth may be visible around lesion edges."
        ),
        "monitor_en": (
            "Inspect frequently during cool, wet, or humid weather. Check whether "
            "lesions expand rapidly or appear on multiple plants."
        ),
        "immediate_en": (
            "Treat rapidly spreading blight-like symptoms as urgent: mark affected "
            "areas, minimize movement through wet foliage, and obtain local diagnostic advice."
        ),
        "prevention_en": (
            "Use healthy seed, remove volunteer plants, improve airflow where possible, "
            "and follow local late-blight monitoring and management programmes."
        ),
        "escalate_en": (
            "Contact a local agricultural extension officer or plant-health professional "
            "promptly if late blight is suspected, especially during favourable weather."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__bacterial_spot",
        "crop_en": "Tomato",
        "condition_en": "Bacterial spot",
        "symptoms_en": (
            "Small dark, sometimes water-soaked spots may occur on leaves. Severe "
            "symptoms can cause yellowing and defoliation."
        ),
        "monitor_en": (
            "Check new growth, nearby plants, and whether symptoms increase after wet weather."
        ),
        "immediate_en": (
            "Avoid handling wet plants, sanitize tools, reduce leaf wetness, and avoid "
            "moving affected plant material between growing areas."
        ),
        "prevention_en": (
            "Use clean seed or transplants, practice crop rotation, maintain sanitation, "
            "and improve airflow and drainage."
        ),
        "escalate_en": (
            "Ask local extension staff for confirmation if symptoms are spreading or "
            "occur widely in young plants."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__early_blight",
        "crop_en": "Tomato",
        "condition_en": "Early blight",
        "symptoms_en": (
            "Brown lesions with concentric ring patterns may develop first on older leaves, "
            "often with yellowing around the lesion."
        ),
        "monitor_en": (
            "Inspect lower, older foliage and check whether spots increase after warm, wet conditions."
        ),
        "immediate_en": (
            "Remove severely affected foliage only when safe and practical, reduce leaf wetness, "
            "and avoid working when foliage is wet."
        ),
        "prevention_en": (
            "Use crop rotation, clean stakes and tools, mulching where appropriate, suitable spacing, "
            "and balanced nutrition."
        ),
        "escalate_en": (
            "Seek local advice if symptoms expand rapidly or affect a substantial portion of the canopy."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__healthy",
        "crop_en": "Tomato",
        "condition_en": "No obvious disease pattern detected",
        "symptoms_en": (
            "The prediction is consistent with a healthy tomato leaf, but it is not a confirmed diagnosis."
        ),
        "monitor_en": (
            "Inspect multiple plants, including lower leaf surfaces, stems, and new growth."
        ),
        "immediate_en": (
            "Continue regular scouting and maintain hygiene, irrigation, and nutrition practices."
        ),
        "prevention_en": (
            "Use clean planting material, sanitize tools, maintain spacing and airflow, "
            "and manage weeds and crop debris."
        ),
        "escalate_en": (
            "Seek support if spots, wilting, yellowing, or rapid symptom spread occurs."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__late_blight",
        "crop_en": "Tomato",
        "condition_en": "Late blight",
        "symptoms_en": (
            "Irregular dark, water-soaked-looking leaf lesions can expand rapidly, "
            "especially during cool and humid weather."
        ),
        "monitor_en": (
            "Check frequently during wet or humid periods and inspect neighbouring tomato and potato plants."
        ),
        "immediate_en": (
            "Rapidly expanding blight-like symptoms require prompt local assessment. "
            "Avoid moving through wet foliage and avoid transporting affected material."
        ),
        "prevention_en": (
            "Use healthy transplants, improve airflow, remove volunteer hosts, practice sanitation, "
            "and follow local disease alerts where available."
        ),
        "escalate_en": (
            "Contact local extension or plant-health services promptly if late blight is suspected."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__leaf_mold",
        "crop_en": "Tomato",
        "condition_en": "Leaf mold",
        "symptoms_en": (
            "Yellowish areas may form on upper leaf surfaces, while olive-green to brown "
            "mold-like growth can occur on lower leaf surfaces in humid conditions."
        ),
        "monitor_en": (
            "Inspect lower leaf surfaces and note whether symptoms are concentrated in humid, dense canopy areas."
        ),
        "immediate_en": (
            "Improve airflow, reduce humidity where possible, and avoid prolonged leaf wetness."
        ),
        "prevention_en": (
            "Use appropriate plant spacing, ventilation in protected cultivation, sanitation, "
            "and locally suitable resistant varieties where available."
        ),
        "escalate_en": (
            "Seek confirmation if symptoms persist, spread in protected cultivation, or affect many plants."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__septoria_leaf_spot",
        "crop_en": "Tomato",
        "condition_en": "Septoria leaf spot",
        "symptoms_en": (
            "Numerous small circular spots with pale or gray centres and darker margins "
            "may develop, often first on lower leaves."
        ),
        "monitor_en": (
            "Inspect lower leaves and monitor whether the number of spots is increasing "
            "after rainfall or overhead irrigation."
        ),
        "immediate_en": (
            "Remove heavily affected lower leaves only if practical, improve spacing and airflow, "
            "and reduce leaf wetness."
        ),
        "prevention_en": (
            "Practice crop rotation, remove crop debris, control volunteer tomatoes and related weeds, "
            "and sanitize tools."
        ),
        "escalate_en": (
            "Seek local support when symptoms spread quickly or lower-canopy defoliation becomes substantial."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__spider_mites_two_spotted_spider_mite",
        "crop_en": "Tomato",
        "condition_en": "Two-spotted spider mite damage",
        "symptoms_en": (
            "Fine stippling, yellowing, bronzing, and sometimes webbing may occur, "
            "especially in hot, dry conditions."
        ),
        "monitor_en": (
            "Inspect leaf undersides with a hand lens if available and look for mites, eggs, and fine webbing."
        ),
        "immediate_en": (
            "Monitor affected areas closely, reduce plant stress, and avoid actions that disrupt beneficial insects."
        ),
        "prevention_en": (
            "Maintain plant health, monitor hot and dry areas, manage dust where feasible, "
            "and use integrated pest management practices."
        ),
        "escalate_en": (
            "Consult local advisers if mite populations increase rapidly or webbing and bronzing become widespread."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__target_spot",
        "crop_en": "Tomato",
        "condition_en": "Target spot",
        "symptoms_en": (
            "Brown lesions with concentric or target-like patterns may occur on leaves. "
            "Symptoms can overlap visually with other tomato foliar diseases."
        ),
        "monitor_en": (
            "Check whether lesions are increasing, merging, or causing leaf yellowing and drop."
        ),
        "immediate_en": (
            "Reduce prolonged leaf wetness, improve airflow, and avoid spreading plant debris or wet foliage."
        ),
        "prevention_en": (
            "Use sanitation, crop rotation, balanced nutrition, and locally appropriate spacing and irrigation practices."
        ),
        "escalate_en": (
            "Request local confirmation because target spot can resemble early blight and other foliar diseases."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__tomato_mosaic_virus",
        "crop_en": "Tomato",
        "condition_en": "Tomato mosaic virus",
        "symptoms_en": (
            "Leaves may show mosaic-like light and dark green patterning, distortion, "
            "or reduced vigour. Symptoms can resemble nutritional or other viral problems."
        ),
        "monitor_en": (
            "Inspect several plants for a repeated mosaic pattern, leaf distortion, "
            "and symptom distribution within the crop."
        ),
        "immediate_en": (
            "Avoid handling plants unnecessarily, wash hands, sanitize tools, and separate "
            "suspect plants while seeking confirmation."
        ),
        "prevention_en": (
            "Use clean seed or transplants, sanitize tools and hands, control volunteer plants, "
            "and use resistant varieties where locally suitable."
        ),
        "escalate_en": (
            "Seek expert confirmation because virus-like symptoms can be confused with other stresses "
            "and management depends on accurate diagnosis."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
    {
        "label": "tomato__tomato_yellow_leaf_curl_virus",
        "crop_en": "Tomato",
        "condition_en": "Tomato yellow leaf curl virus",
        "symptoms_en": (
            "Young leaves may curl upward, become smaller, and show yellowing. Plants may become stunted."
        ),
        "monitor_en": (
            "Inspect new growth and monitor for whitefly activity, especially around crop edges and nearby hosts."
        ),
        "immediate_en": (
            "Isolate suspect plants where practical, manage weeds and volunteer hosts, and seek local confirmation."
        ),
        "prevention_en": (
            "Use clean transplants, resistant varieties where available, physical exclusion in protected systems, "
            "and integrated whitefly monitoring and management."
        ),
        "escalate_en": (
            "Contact local extension staff if virus-like symptoms or whitefly activity are increasing."
        ),
        "source": "Curated IPM advisory knowledge base",
    },
]

knowledge_base_df = pd.DataFrame(knowledge_base)

assert len(knowledge_base_df) == 19
assert set(knowledge_base_df["label"]) == set(MAIN19_LABELS)

display(
    knowledge_base_df[
        [
            "label",
            "crop_en",
            "condition_en",
            "source",
        ]
    ]
)

knowledge_base_df.to_csv(
    RESULTS_DIR / "knowledge_base_main19.csv",
    index=False,
)

In [ ]:
def build_retrieval_document(record):
    return " ".join(
        [
            record["label"].replace("__", " "),
            record["crop_en"],
            record["condition_en"],
            record["symptoms_en"],
            record["monitor_en"],
            record["immediate_en"],
            record["prevention_en"],
            record["escalate_en"],
        ]
    )

knowledge_base_df["retrieval_document"] = (
    knowledge_base_df
    .apply(
        build_retrieval_document,
        axis=1,
    )
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
)

knowledge_matrix = vectorizer.fit_transform(
    knowledge_base_df["retrieval_document"]
)

print("Knowledge-base entries:", len(knowledge_base_df))
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("TF-IDF matrix shape:", knowledge_matrix.shape)

In [ ]:
def retrieve_advisory(
    query,
    predicted_label=None,
    top_k=3,
):
    query = str(query).strip()

    query_vector = vectorizer.transform([query])

    semantic_scores = cosine_similarity(
        query_vector,
        knowledge_matrix,
    ).flatten()

    retrieval_df = knowledge_base_df[
        [
            "label",
            "crop_en",
            "condition_en",
            "symptoms_en",
            "monitor_en",
            "immediate_en",
            "prevention_en",
            "escalate_en",
            "source",
        ]
    ].copy()

    retrieval_df["semantic_score"] = semantic_scores

    # Prioritize the classifier prediction when it is in the Main19 KB.
    # The user query still determines the remaining rank order.
    if predicted_label in set(knowledge_base_df["label"]):
        retrieval_df["prediction_bonus"] = np.where(
            retrieval_df["label"] == predicted_label,
            1.0,
            0.0,
        )

        retrieval_df["combined_score"] = (
            0.70 * retrieval_df["prediction_bonus"]
            + 0.30 * retrieval_df["semantic_score"]
        )

    else:
        retrieval_df["prediction_bonus"] = 0.0
        retrieval_df["combined_score"] = (
            retrieval_df["semantic_score"]
        )

    retrieval_df = (
        retrieval_df
        .sort_values(
            "combined_score",
            ascending=False,
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    return retrieval_df


def retrieve_by_prediction(
    predicted_label,
    top_k=3,
):
    crop, disease = split_main19_label(predicted_label)

    query = (
        f"{crop} {disease.replace('_', ' ')} "
        "symptoms monitoring immediate actions prevention expert advice"
    )

    return retrieve_advisory(
        query=query,
        predicted_label=predicted_label,
        top_k=top_k,
    )

In [ ]:
test_queries = [
    {
        "query": "Tomato leaves have small circular spots with pale centers. What should I do?",
        "predicted_label": "tomato__septoria_leaf_spot",
    },
    {
        "query": "Potato leaves have rapidly spreading dark water-soaked lesions after humid weather.",
        "predicted_label": "potato__late_blight",
    },
    {
        "query": "Corn leaves show long cigar-shaped brown lesions.",
        "predicted_label": "corn__northern_leaf_blight",
    },
]

for example in test_queries:
    print("\nQuery:", example["query"])
    print("Model prediction:", example["predicted_label"])

    retrieved = retrieve_advisory(
        query=example["query"],
        predicted_label=example["predicted_label"],
        top_k=3,
    )

    display(
        retrieved[
            [
                "label",
                "condition_en",
                "combined_score",
                "semantic_score",
            ]
        ]
    )

In [ ]:
# Compact, reviewable translations for project demonstration.
# Disease names are retained in English where terminology may vary locally.
# This avoids unsafe or misleading automatic medical/agronomic translation.

TRANSLATIONS = {
    "en": {
        "title": "AI-assisted crop advisory",
        "preliminary": "Preliminary image-model result",
        "confidence": "Model confidence",
        "reliability": "Reliability level",
        "condition": "Retrieved condition",
        "symptoms": "Typical visible signs",
        "monitor": "What to check next",
        "immediate": "Immediate low-risk actions",
        "prevention": "Prevention and field management",
        "escalate": "When to seek expert help",
        "safety": (
            "This is an AI-assisted preliminary result, not a confirmed diagnosis. "
            "Image-only predictions can be unreliable under field conditions. "
            "Do not make pesticide decisions from this result alone; follow local regulations "
            "and seek advice from a qualified agricultural extension or plant-health professional."
        ),
        "low_confidence": (
            "The prediction confidence is low. Please take clearer images of multiple leaves "
            "from both sides, include the whole plant and nearby symptoms, and request expert confirmation."
        ),
        "medium_confidence": (
            "The prediction has moderate confidence. Treat the result as a screening signal and "
            "inspect multiple plants before acting."
        ),
        "high_confidence": (
            "The prediction has high model confidence, but it still requires field verification "
            "because confidence does not guarantee correctness under real-world conditions."
        ),
        "retrieved_sources": "Retrieved knowledge entries",
    },
    "hi": {
        "title": "एआई-सहायित फसल सलाह",
        "preliminary": "प्रारंभिक इमेज-मॉडल परिणाम",
        "confidence": "मॉडल विश्वास",
        "reliability": "विश्वसनीयता स्तर",
        "condition": "प्राप्त स्थिति",
        "symptoms": "दिखने वाले सामान्य लक्षण",
        "monitor": "आगे क्या जांचें",
        "immediate": "तत्काल कम-जोखिम वाले कदम",
        "prevention": "रोकथाम और खेत प्रबंधन",
        "escalate": "विशेषज्ञ सहायता कब लें",
        "safety": (
            "यह एआई-सहायित प्रारंभिक परिणाम है, पुष्टि किया गया निदान नहीं। "
            "खेत की वास्तविक परिस्थितियों में केवल तस्वीर के आधार पर अनुमान गलत हो सकता है। "
            "केवल इस परिणाम के आधार पर कीटनाशक का निर्णय न लें; स्थानीय नियमों का पालन करें "
            "और कृषि विस्तार अधिकारी या पौध-स्वास्थ्य विशेषज्ञ से सलाह लें।"
        ),
        "low_confidence": (
            "मॉडल का विश्वास कम है। दोनों तरफ से कई पत्तियों की स्पष्ट तस्वीर लें, "
            "पूरा पौधा और आसपास के लक्षण दिखाएं, तथा विशेषज्ञ से पुष्टि कराएं।"
        ),
        "medium_confidence": (
            "मॉडल का विश्वास मध्यम है। इसे केवल प्रारंभिक संकेत मानें और कार्रवाई से पहले "
            "कई पौधों की जांच करें।"
        ),
        "high_confidence": (
            "मॉडल का विश्वास अधिक है, लेकिन खेत की स्थिति में परिणाम की पुष्टि आवश्यक है। "
            "अधिक विश्वास का अर्थ हमेशा सही निदान नहीं होता।"
        ),
        "retrieved_sources": "प्राप्त ज्ञान प्रविष्टियां",
    },
    "te": {
        "title": "ఏఐ ఆధారిత పంట సలహా",
        "preliminary": "ప్రాథమిక ఇమేజ్-మోడల్ ఫలితం",
        "confidence": "మోడల్ నమ్మకం",
        "reliability": "విశ్వసనీయత స్థాయి",
        "condition": "సేకరించిన పరిస్థితి",
        "symptoms": "కనిపించే సాధారణ లక్షణాలు",
        "monitor": "తదుపరి ఏమి పరిశీలించాలి",
        "immediate": "తక్షణ తక్కువ-ప్రమాద చర్యలు",
        "prevention": "నివారణ మరియు పొల నిర్వహణ",
        "escalate": "నిపుణుల సహాయం ఎప్పుడు తీసుకోవాలి",
        "safety": (
            "ఇది ఏఐ ఆధారిత ప్రాథమిక ఫలితం మాత్రమే; నిర్ధారిత నిర్ధారణ కాదు. "
            "పొలంలోని పరిస్థితుల్లో కేవలం చిత్రం ఆధారంగా అంచనా తప్పు కావచ్చు. "
            "ఈ ఫలితంపై మాత్రమే పురుగుమందుల నిర్ణయం తీసుకోకండి; స్థానిక నిబంధనలు పాటించి "
            "వ్యవసాయ విస్తరణ అధికారి లేదా మొక్కల ఆరోగ్య నిపుణుడి సలహా తీసుకోండి."
        ),
        "low_confidence": (
            "మోడల్ నమ్మకం తక్కువగా ఉంది. రెండు వైపులా అనేక ఆకుల స్పష్టమైన చిత్రాలు తీసి, "
            "మొత్తం మొక్క మరియు చుట్టుపక్కల లక్షణాలను చూపించి నిపుణుడి నిర్ధారణ పొందండి."
        ),
        "medium_confidence": (
            "మోడల్ నమ్మకం మధ్యస్థంగా ఉంది. దీనిని ప్రాథమిక సూచనగా మాత్రమే పరిగణించి, "
            "చర్య తీసుకునే ముందు అనేక మొక్కలను పరిశీలించండి."
        ),
        "high_confidence": (
            "మోడల్ నమ్మకం ఎక్కువగా ఉంది, అయినప్పటికీ పొల స్థాయి ధృవీకరణ అవసరం. "
            "ఎక్కువ నమ్మకం ఎల్లప్పుడూ సరైన నిర్ధారణను సూచించదు."
        ),
        "retrieved_sources": "సేకరించిన జ్ఞాన ప్రవేశాలు",
    },
}

In [ ]:
def confidence_band(confidence):
    confidence = float(confidence)

    if confidence < MEDIUM_CONFIDENCE_THRESHOLD:
        return "low"

    if confidence < HIGH_CONFIDENCE_THRESHOLD:
        return "medium"

    return "high"


def get_language_text(language_code):
    if language_code not in TRANSLATIONS:
        raise ValueError(
            f"Unsupported language: {language_code}. "
            f"Choose from {list(TRANSLATIONS.keys())}."
        )

    return TRANSLATIONS[language_code]


def format_advisory(
    predicted_label,
    confidence,
    user_query="",
    language="en",
    top_k=3,
):
    if predicted_label not in set(knowledge_base_df["label"]):
        raise ValueError(
            f"Unknown Main19 label: {predicted_label}"
        )

    language_text = get_language_text(language)

    retrieved_entries = retrieve_advisory(
        query=user_query or predicted_label.replace("__", " "),
        predicted_label=predicted_label,
        top_k=top_k,
    )

    primary_entry = retrieved_entries.iloc[0].to_dict()

    band = confidence_band(confidence)

    confidence_message_key = f"{band}_confidence"

    report_lines = [
        f"# {language_text['title']}",
        "",
        f"**{language_text['preliminary']}:** "
        f"{primary_entry['crop_en']} — {primary_entry['condition_en']}",
        "",
        f"**{language_text['confidence']}:** {float(confidence):.1%}",
        "",
        f"**{language_text['reliability']}:** {band.upper()}",
        "",
        f"**{language_text['symptoms']}:** "
        f"{primary_entry['symptoms_en']}",
        "",
        f"**{language_text['monitor']}:** "
        f"{primary_entry['monitor_en']}",
        "",
        f"**{language_text['immediate']}:** "
        f"{primary_entry['immediate_en']}",
        "",
        f"**{language_text['prevention']}:** "
        f"{primary_entry['prevention_en']}",
        "",
        f"**{language_text['escalate']}:** "
        f"{primary_entry['escalate_en']}",
        "",
        f"**Safety note:** {language_text['safety']}",
        "",
        f"**Confidence guidance:** "
        f"{language_text[confidence_message_key]}",
        "",
        f"**{language_text['retrieved_sources']}:**",
    ]

    for rank, (_, entry) in enumerate(
        retrieved_entries.iterrows(),
        start=1,
    ):
        report_lines.append(
            f"{rank}. {entry['crop_en']} — {entry['condition_en']} "
            f"(score: {entry['combined_score']:.3f}; "
            f"source: {entry['source']})"
        )

    return {
        "advisory_markdown": "\n".join(report_lines),
        "confidence_band": band,
        "primary_entry": primary_entry,
        "retrieved_entries": retrieved_entries,
    }

In [ ]:
english_example = format_advisory(
    predicted_label="potato__late_blight",
    confidence=0.82,
    user_query=(
        "My potato leaves have dark spreading patches after wet weather. "
        "What should I inspect and do next?"
    ),
    language="en",
    top_k=3,
)

print(english_example["advisory_markdown"])

display(
    english_example["retrieved_entries"][
        [
            "label",
            "condition_en",
            "combined_score",
            "semantic_score",
        ]
    ]
)

In [ ]:
hindi_example = format_advisory(
    predicted_label="tomato__early_blight",
    confidence=0.74,
    user_query=(
        "Tomato leaves show brown spots with rings. "
        "What should I check?"
    ),
    language="hi",
    top_k=3,
)

print(hindi_example["advisory_markdown"])

In [ ]:
telugu_example = format_advisory(
    predicted_label=(
        "tomato__tomato_yellow_leaf_curl_virus"
    ),
    confidence=0.58,
    user_query=(
        "Tomato young leaves are curling and yellowing. "
        "What should I do?"
    ),
    language="te",
    top_k=3,
)

print(telugu_example["advisory_markdown"])

In [ ]:
def generate_advisory_from_prediction(
    prediction_result,
    language="en",
    user_query="",
):
    required_fields = {
        "predicted_label",
        "confidence",
    }

    missing_fields = (
        required_fields
        - set(prediction_result.keys())
    )

    if missing_fields:
        raise ValueError(
            f"prediction_result is missing: {missing_fields}"
        )

    predicted_label = prediction_result["predicted_label"]
    confidence = float(prediction_result["confidence"])

    advisory = format_advisory(
        predicted_label=predicted_label,
        confidence=confidence,
        user_query=user_query,
        language=language,
        top_k=3,
    )

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "confidence_band": advisory["confidence_band"],
        "advisory_markdown": advisory["advisory_markdown"],
        "retrieved_entries": advisory["retrieved_entries"],
    }


sample_prediction_result = {
    "predicted_label": "corn__common_rust",
    "confidence": 0.93,
}

sample_advisory = generate_advisory_from_prediction(
    prediction_result=sample_prediction_result,
    language="en",
    user_query=(
        "Corn leaves have orange-brown raised spots. "
        "Give safe next steps."
    ),
)

print(sample_advisory["advisory_markdown"])

In [ ]:
validation_rows = []

for label in MAIN19_LABELS:
    result = format_advisory(
        predicted_label=label,
        confidence=0.85,
        user_query=label.replace("__", " "),
        language="en",
        top_k=3,
    )

    primary_label = result["primary_entry"]["label"]

    validation_rows.append(
        {
            "input_label": label,
            "retrieved_primary_label": primary_label,
            "primary_retrieval_matches_prediction": (
                label == primary_label
            ),
            "confidence_band": result["confidence_band"],
            "advisory_length_characters": len(
                result["advisory_markdown"]
            ),
        }
    )

validation_df = pd.DataFrame(validation_rows)

display(validation_df)

assert validation_df[
    "primary_retrieval_matches_prediction"
].all(), (
    "Some Main19 labels did not retrieve their own primary knowledge entry."
)

validation_df.to_csv(
    RESULTS_DIR / "table_01_rag_validation_all_main19_labels.csv",
    index=False,
)

In [ ]:
confidence_test_values = [
    0.35,
    0.69,
    0.70,
    0.89,
    0.90,
    0.99,
]

confidence_policy_df = pd.DataFrame(
    {
        "confidence": confidence_test_values,
        "advisory_band": [
            confidence_band(value)
            for value in confidence_test_values
        ],
        "expected_behavior": [
            (
                "Request clearer images and expert confirmation"
                if confidence_band(value) == "low"
                else (
                    "Treat as screening signal; inspect multiple plants"
                    if confidence_band(value) == "medium"
                    else (
                        "Provide guidance with mandatory field-verification warning"
                    )
                )
            )
            for value in confidence_test_values
        ],
    }
)

display(confidence_policy_df)

confidence_policy_df.to_csv(
    RESULTS_DIR / "table_02_confidence_safety_policy.csv",
    index=False,
)

In [ ]:
low_confidence_example = format_advisory(
    predicted_label="tomato__leaf_mold",
    confidence=0.48,
    user_query=(
        "There are yellow patches on tomato leaves, but the image is unclear."
    ),
    language="en",
    top_k=3,
)

print(low_confidence_example["advisory_markdown"])

In [ ]:
advisory_card_rows = []

for label in MAIN19_LABELS:
    entry = knowledge_base_df.loc[
        knowledge_base_df["label"] == label
    ].iloc[0]

    advisory_card_rows.append(
        {
            "label": label,
            "crop": entry["crop_en"],
            "condition": entry["condition_en"],
            "monitoring_focus": entry["monitor_en"],
            "immediate_low_risk_actions": entry["immediate_en"],
            "prevention": entry["prevention_en"],
            "expert_escalation": entry["escalate_en"],
        }
    )

advisory_cards_df = pd.DataFrame(
    advisory_card_rows
)

display(advisory_cards_df)

advisory_cards_df.to_csv(
    RESULTS_DIR / "table_03_main19_advisory_cards.csv",
    index=False,
)

In [ ]:
run_metadata = {
    "notebook": "11_rag_multilingual_advisory",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "knowledge_base_entries": int(
        len(knowledge_base_df)
    ),
    "expected_main19_labels": int(
        len(MAIN19_LABELS)
    ),
    "retrieval_method": (
        "TF-IDF cosine similarity with classifier-label prioritization"
    ),
    "supported_languages": SUPPORTED_LANGUAGES,
    "medium_confidence_threshold": (
        MEDIUM_CONFIDENCE_THRESHOLD
    ),
    "high_confidence_threshold": (
        HIGH_CONFIDENCE_THRESHOLD
    ),
    "safety_policy": (
        "Low confidence requests expert confirmation; medium confidence "
        "is screening-only; high confidence still includes mandatory "
        "field-verification and no-pesticide-decision warning."
    ),
    "scope_limitations": (
        "Educational advisory only. No pesticide brand, dosage, or "
        "location-specific chemical recommendation. Model output is not "
        "a confirmed diagnosis, particularly under external-domain conditions."
    ),
}

with open(
    RESULTS_DIR / "run_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        run_metadata,
        indent=2,
        ensure_ascii=False,
    )
)

## Notebook 11 conclusion

This notebook implements a retrieval-augmented multilingual advisory layer for Main19 classifier outputs.

### What it provides

- A curated knowledge base for all 19 Main19 crop–disease labels.
- TF-IDF retrieval with classifier-label prioritization.
- Structured advisory sections: visible signs, monitoring, low-risk actions,
  prevention, and escalation.
- English, Hindi, and Telugu advisory templates.
- A confidence-based safety policy.
- Explicit warnings that image-model output is preliminary and requires field verification.

### Important limitation

The advisory layer does not make the classifier more accurate. Notebook 10 showed substantial external domain shift on PlantDoc. Therefore, retrieval guidance must be presented as conditional support after a preliminary prediction, not as a confirmed diagnosis or pesticide prescription.

### Suggested report statement

> “A retrieval-augmented multilingual advisory module was developed to convert model outputs into structured, safety-aware crop-management guidance. The module retrieves a curated crop–disease entry and presents monitoring, sanitation, prevention, and escalation advice in English, Hindi, and Telugu. Because external-domain evaluation showed substantial prediction uncertainty, every advisory includes a field-verification and expert-consultation warning.”